# 03 — End-to-End Smoke Test

> **Phase:** 2 — gate before any Phase-4 experiment touches the full 12,723

**Why this notebook exists.** Phase 4 will run this exact pipeline ~63,615 times (5 architectures × 12,723 questions). If anything is wrong — prompt format, retrieval prefix, letter parsing, Groq wiring — discover it now on **3 calls** instead of after 30 hours of compute.

**Pipeline tested:**

```
question (MedQA dev)
  └─► embed_queries (BGE prefix)   ← src/data/embedder.py
        └─► ChromaDB top-5         ← src/data/indices.py
              └─► build_evidence_grounded_prompt   ← src/generation/prompts.py
                    └─► groq_complete (cached)     ← src/generation/groq_client.py
                          └─► parse_letter         ← src/generation/prompts.py
```

**Acceptance ([docs/todo.md §2.3](../docs/todo.md)):**
- 3 sensible answer rows on screen
- Average latency < 5 s per question
- All three predicted letters parse cleanly (A/B/C/D)

**Cost.** 3 Groq calls on first run (LLaMA 3.3 70B is on Groq's free tier). All re-runs hit the disk cache → $0.

## 1. Setup

In [2]:
import sys, os, json, time, textwrap, logging
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Quiet chromadb's posthog telemetry
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)

cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Load .env for API keys
load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import load_bge, embed_queries, BGE_QUERY_PREFIX, best_device
from src.data.indices import load_chroma
from src.generation.prompts import build_evidence_grounded_prompt, parse_letter
from src.generation.groq_client import groq_complete, DEFAULT_MODEL

CHUNKS_PATH       = REPO_ROOT / "data" / "processed" / "chunks.parquet"
MEDQA_4OPT_PATH   = REPO_ROOT / "data" / "processed" / "medqa_4opt.parquet"
CHROMA_DIR        = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
CACHE_DIR         = REPO_ROOT / "data" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:    {REPO_ROOT}")
print(f"Device:       {best_device()}")
print(f"Groq model:   {DEFAULT_MODEL}")
print(f"Cache dir:    {CACHE_DIR}")


Repo root:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Device:       mps
Groq model:   llama-3.3-70b-versatile
Cache dir:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/cache


## 2. Verify `GROQ_API_KEY`

If this cell fails, populate `.env` at the repo root with:

```
GROQ_API_KEY=gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

…then re-run from cell §1. The other two keys (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`) are needed for Phase 3 and Phase 4 RAGAS judging respectively — they don't block this notebook.

In [3]:
groq_key = os.environ.get("GROQ_API_KEY")
assert groq_key, (
    "GROQ_API_KEY missing. Add it to .env at the repo root and rerun §1+§2.\n"
    "Sign up + grab a key: https://console.groq.com/keys"
)
print(f"GROQ_API_KEY: present (length {len(groq_key)})")
print(f"Other keys:")
for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]:
    v = os.environ.get(k)
    print(f"  {k:<22}  {'present' if v else 'missing (not blocking — needed in Phase 3/4)'}")


GROQ_API_KEY: present (length 56)
Other keys:
  OPENAI_API_KEY          missing (not blocking — needed in Phase 3/4)
  ANTHROPIC_API_KEY       missing (not blocking — needed in Phase 3/4)


## 3. Load shared infrastructure from disk

Loads `chunks.parquet` + ChromaDB collection + BGE model. ChromaDB and BGE were both built in Notebook 02; this notebook should never re-embed or re-index.

In [4]:
# Chunks lookup (chunk_id → text) for displaying retrieved passages
chunks_df = pd.read_parquet(CHUNKS_PATH)
chunk_text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))

# ChromaDB collection (pre-built in Notebook 02)
chroma_coll = load_chroma(CHROMA_DIR)
assert chroma_coll.count() == len(chunks_df), \
    f"Chroma count {chroma_coll.count()} != chunks {len(chunks_df)} — re-run Notebook 02"

# BGE model (only used for query encoding — chunk encoding was done in Notebook 02)
%time model = load_bge(device=best_device())

print(f"\nchunks_df rows  : {len(chunks_df):,}")
print(f"ChromaDB count  : {chroma_coll.count():,}")
print(f"BGE device      : {model.device}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


CPU times: user 203 ms, sys: 561 ms, total: 765 ms
Wall time: 9.43 s

chunks_df rows  : 67,599
ChromaDB count  : 67,599
BGE device      : mps:0


## 4. Pick 3 dev questions

Stratified random pick from the **dev split** of `medqa_4opt.parquet` (1,272 questions). Seed 42 for reproducibility — same three questions every run. The 4-option variant carries `metamap_phrases` and is what every later experiment will use.

In [5]:
medqa = pd.read_parquet(MEDQA_4OPT_PATH)
dev_sample = (
    medqa[medqa["split"] == "dev"]
    .sample(3, random_state=42)
    .reset_index(drop=True)
)

for i, row in dev_sample.iterrows():
    opts = json.loads(row["options_json"])
    print(f"--- dev[{i}]  meta_info={row['meta_info']}  truth={row['answer_idx']} ---")
    print(textwrap.fill(row["question"], width=110))
    print()
    for k in sorted(opts):
        print(f"  {k}) {opts[k]}")
    print()


--- dev[0]  meta_info=step1  truth=B ---
A 67-year-old woman comes to the physician because of a 5-day history of episodic abdominal pain, nausea, and
vomiting. She has coronary artery disease and type 2 diabetes mellitus. She takes aspirin, metoprolol, and
metformin. She is 163 cm (5 ft 4 in) tall and weighs 91 kg (200 lb); her BMI is 34 kg/m2. Her temperature is
38.1°C (100.6°F). Physical examination shows dry mucous membranes, abdominal distension, and hyperactive bowel
sounds. Ultrasonography of the abdomen shows air in the biliary tract. This patient's symptoms are most likely
caused by obstruction at which of the following locations?

  A) Third part of the duodenum
  B) Distal ileum
  C) Proximal jejunum
  D) Pancreatic duct

--- dev[1]  meta_info=step2&3  truth=A ---
A 29-year-old man comes into his primary care physician's office with a chief complaint of a cough. The
patient states that the cough started yesterday and he is asking if he needs antibiotics. While conversing
wit

## 5. Define the per-question pipeline

One helper that does the whole flow for a single row. Returns a dict so we can build a results table.

In [6]:
def answer_question(row, k: int = 5) -> dict:
    """Run question → retrieve top-k → prompt → Groq → parse, return audit dict."""
    options = json.loads(row["options_json"])

    # 1. Embed the query (BGE prefix is added inside embed_queries)
    q_emb = embed_queries(model, [row["question"]])

    # 2. ChromaDB top-k
    res = chroma_coll.query(query_embeddings=q_emb.tolist(), n_results=k)
    retrieved_ids   = res["ids"][0]
    retrieved_texts = res["documents"][0]
    retrieved_dists = res["distances"][0]

    # 3. Build the evidence-grounded prompt
    prompt = build_evidence_grounded_prompt(row["question"], options, retrieved_texts)

    # 4. Cached Groq call
    text, latency_s, was_cached = groq_complete(
        prompt=prompt, cache_dir=CACHE_DIR,
    )

    # 5. Parse the predicted letter
    predicted = parse_letter(text)
    truth     = row["answer_idx"]

    return {
        "question":       row["question"],
        "options":        options,
        "truth":          truth,
        "predicted":      predicted,
        "correct":        predicted == truth,
        "raw_response":   text,
        "latency_s":      latency_s,
        "was_cached":     was_cached,
        "retrieved_ids":  retrieved_ids,
        "retrieved_dists": retrieved_dists,
        "retrieved_texts": retrieved_texts,
        "prompt":         prompt,
    }


## 6. Tiny smoke — run question #1 first

Run a single question to confirm: prompt renders, Groq returns text, parser extracts a letter. ~1 s if cached, ~2-4 s if first call.

In [7]:
smoke = answer_question(dev_sample.iloc[0])

print(f"truth: {smoke['truth']}   predicted: {smoke['predicted']}   correct: {smoke['correct']}")
print(f"latency: {smoke['latency_s']:.2f} s   cached: {smoke['was_cached']}")
print(f"raw response: {smoke['raw_response']!r}")
print(f"\n--- prompt (first 600 chars of {len(smoke['prompt'])}) ---")
print(smoke["prompt"][:600])
print("...")

assert smoke["predicted"] in {"A", "B", "C", "D", "E"}, \
    f"parser failed to extract a letter from {smoke['raw_response']!r}"
print("\n✓ smoke OK — prompt renders, Groq returns text, parser extracts a valid letter")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


truth: B   predicted: B   correct: True
latency: 0.35 s   cached: False
raw response: 'B'

--- prompt (first 600 chars of 9111) ---
You are a medical expert answering a USMLE multiple-choice question.

Use ONLY the evidence below to choose the best option. If the evidence is insufficient, still pick the single most plausible option based on the question. Do not explain your reasoning.

EVIDENCE:
[1] Bowel sounds are hypoactive and are substantially decreased with peritonitis related to a ruptured diverticular abscess. Abdominal examination reveals distention with left lower quadrant tenderness on direct palpation and localized rebound tenderness. Abdominal and bimanual rectovaginal examinations may reveal a poorly mobile, 
...

✓ smoke OK — prompt renders, Groq returns text, parser extracts a valid letter


## 7. Run all 3 questions

In [8]:
results = [answer_question(row) for _, row in dev_sample.iterrows()]
print(f"Ran {len(results)} questions.")


Ran 3 questions.


## 8. Side-by-side display

For each question: question text, top-5 retrieved chunks (id + first 200 chars), the LLM's raw response, parsed letter vs ground truth, latency.

In [11]:
WIDTH = 110

for i, r in enumerate(results):
    badge = "✓ CORRECT" if r["correct"] else "✗ wrong"
    print("=" * WIDTH)
    print(f"QUESTION {i}   truth={r['truth']}  predicted={r['predicted']}  {badge}  "
          f"latency={r['latency_s']:.2f}s  cached={r['was_cached']}")
    print("=" * WIDTH)
    print(textwrap.fill(r["question"], width=WIDTH))
    print()
    for k in sorted(r["options"]):
        marker = "  ← truth" if k == r["truth"] else ("  ← pred" if k == r["predicted"] else "")
        print(f"  {k}) {r['options'][k]}{marker}")
    print(f"\n--- Top-5 retrieved chunks ---")
    for cid, dist, text in zip(r["retrieved_ids"], r["retrieved_dists"], r["retrieved_texts"]):
        snippet = text[:200].replace("\n", " ")
        print(f"  [{cid}] cos_dist={dist:.4f}")
        print(f"    {snippet}...")
    print(f"\n--- LLM raw response ---")
    print(f"  {r['raw_response']!r}")
    print()


QUESTION 0   truth=B  predicted=B  ✓ CORRECT  latency=0.35s  cached=True
A 67-year-old woman comes to the physician because of a 5-day history of episodic abdominal pain, nausea, and
vomiting. She has coronary artery disease and type 2 diabetes mellitus. She takes aspirin, metoprolol, and
metformin. She is 163 cm (5 ft 4 in) tall and weighs 91 kg (200 lb); her BMI is 34 kg/m2. Her temperature is
38.1°C (100.6°F). Physical examination shows dry mucous membranes, abdominal distension, and hyperactive bowel
sounds. Ultrasonography of the abdomen shows air in the biliary tract. This patient's symptoms are most likely
caused by obstruction at which of the following locations?

  A) Third part of the duodenum
  B) Distal ileum  ← truth
  C) Proximal jejunum
  D) Pancreatic duct

--- Top-5 retrieved chunks ---
  [Gynecology_Novak_chunk_01256] cos_dist=0.2380
    Bowel sounds are hypoactive and are substantially decreased with peritonitis related to a ruptured diverticular abscess. Abdominal e

## 9. Acceptance summary

In [ ]:
# Compute summary stats
n           = len(results)
n_parsed    = sum(1 for r in results if r["predicted"] is not None)
n_correct   = sum(1 for r in results if r["correct"])
mean_lat    = float(np.mean([r["latency_s"] for r in results]))
n_cache_hit = sum(1 for r in results if r["was_cached"])

rows = [
    ("Questions run",          f"{n}"),
    ("Letters parsed",         f"{n_parsed} / {n}"),
    ("Correct vs ground truth", f"{n_correct} / {n}"),
    ("Mean latency",           f"{mean_lat:.2f} s"),
    ("Cache hits",             f"{n_cache_hit} / {n}"),
    ("Cache dir",              str(CACHE_DIR.relative_to(REPO_ROOT))),
]
for k, v in rows:
    print(f"  {k:<28} {v}")

# Acceptance: all 3 parse cleanly + mean latency < 5 s
assert n_parsed == n, "letter parser failed on at least one response"
assert mean_lat < 5.0, f"mean latency {mean_lat:.2f}s exceeds 5 s budget"
print("\n✓ acceptance: all letters parsed; mean latency under 5 s")


---

**Done.** The end-to-end retrieve → prompt → Groq → parse pipeline works on real MedQA dev questions.

**What's now safe to do:**
- Phase 3 — golden RAGAS dataset construction (Notebook 04)
- Phase 4 — EXP_01 (No-RAG) → EXP_02 (Naive) → ... → EXP_05 (Multi-Hop) on the full 12,723

**Note on accuracy.** With only 3 questions, *correctness* is just a sanity readout, not a metric. A wrong answer on one of them isn't a failure — the goal is the pipeline plumbing, not the score. Real accuracy numbers come from EXP_01–EXP_05.